In [3]:
import json
import sys
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.preprocessing import clean_transcript, split_speakers, split_sentences

DATA_PATH = BASE_DIR / "data" / "corpus.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(type(corpus))
print(len(corpus["reunions"]))

<class 'dict'>
8


In [4]:
records = []

for meeting_index, meeting in enumerate(corpus["reunions"]):

    for reply_index, reply in enumerate(meeting["repliques"]):

        text = reply.get("texte", "")

        records.append({
            "meeting_id": meeting_index + 1,
            "reply_id": reply_index + 1,
            "text": text
        })

transcripts_df = pd.DataFrame(records)

transcripts_df.head()

,meeting_id,reply_id,text
0,1,1,"Right, let us get started, I have a hard stop ..."
1,1,2,"Two people are off next week, so we will be sh..."
2,1,3,I will update the pricing on the kiosk by Wedn...
3,1,4,I will take down the summer banner.
4,1,5,"Nadia, can you call the sign shop about the aw..."


In [5]:
print("Total transcript segments:", len(transcripts_df))
print("Meetings:", transcripts_df["meeting_id"].nunique())
print("Empty segments:", (transcripts_df["text"].str.strip() == "").sum())

Total transcript segments: 154
Meetings: 8
Empty segments: 0


In [6]:
transcripts_df["clean_text"] = transcripts_df["text"].apply(
    clean_transcript
)

transcripts_df.head()

,meeting_id,reply_id,text,clean_text
0,1,1,"Right, let us get started, I have a hard stop ...","Right, let us get started, I have a hard stop ..."
1,1,2,"Two people are off next week, so we will be sh...","Two people are off next week, so we will be sh..."
2,1,3,I will update the pricing on the kiosk by Wedn...,I will update the pricing on the kiosk by Wedn...
3,1,4,I will take down the summer banner.,I will take down the summer banner.
4,1,5,"Nadia, can you call the sign shop about the aw...","Nadia, can you call the sign shop about the aw..."


In [7]:
transcripts_df["sentences"] = transcripts_df["clean_text"].apply(
    split_sentences
)

transcripts_df["sentence_count"] = transcripts_df["sentences"].apply(
    len
)

transcripts_df.head()

,meeting_id,reply_id,text,clean_text,sentences,sentence_count
0,1,1,"Right, let us get started, I have a hard stop ...","Right, let us get started, I have a hard stop ...","[Right, let us get started, I have a hard stop...",1
1,1,2,"Two people are off next week, so we will be sh...","Two people are off next week, so we will be sh...","[Two people are off next week, so we will be s...",1
2,1,3,I will update the pricing on the kiosk by Wedn...,I will update the pricing on the kiosk by Wedn...,[I will update the pricing on the kiosk by Wed...,1
3,1,4,I will take down the summer banner.,I will take down the summer banner.,[I will take down the summer banner.],1
4,1,5,"Nadia, can you call the sign shop about the aw...","Nadia, can you call the sign shop about the aw...","[Nadia, can you call the sign shop about the a...",1


In [8]:
transcripts_df["speaker_segments"] = transcripts_df["clean_text"].apply(
    split_speakers
)

transcripts_df.head()

,meeting_id,reply_id,text,clean_text,sentences,sentence_count,speaker_segments
0,1,1,"Right, let us get started, I have a hard stop ...","Right, let us get started, I have a hard stop ...","[Right, let us get started, I have a hard stop...",1,"[Right, let us get started, I have a hard stop..."
1,1,2,"Two people are off next week, so we will be sh...","Two people are off next week, so we will be sh...","[Two people are off next week, so we will be s...",1,"[Two people are off next week, so we will be s..."
2,1,3,I will update the pricing on the kiosk by Wedn...,I will update the pricing on the kiosk by Wedn...,[I will update the pricing on the kiosk by Wed...,1,[I will update the pricing on the kiosk by Wed...
3,1,4,I will take down the summer banner.,I will take down the summer banner.,[I will take down the summer banner.],1,[I will take down the summer banner.]
4,1,5,"Nadia, can you call the sign shop about the aw...","Nadia, can you call the sign shop about the aw...","[Nadia, can you call the sign shop about the a...",1,"[Nadia, can you call the sign shop about the a..."


In [9]:
sentence_records = []

for _, row in transcripts_df.iterrows():

    for sentence_id, sentence in enumerate(
        row["sentences"],
        start=1
    ):

        sentence_records.append({
            "meeting_id": row["meeting_id"],
            "reply_id": row["reply_id"],
            "sentence_id": sentence_id,
            "sentence": sentence
        })

sentences_df = pd.DataFrame(sentence_records)

sentences_df.head(10)

,meeting_id,reply_id,sentence_id,sentence
0,1,1,1,"Right, let us get started, I have a hard stop ..."
1,1,2,1,"Two people are off next week, so we will be sh..."
2,1,3,1,I will update the pricing on the kiosk by Wedn...
3,1,4,1,I will take down the summer banner.
4,1,5,1,"Nadia, can you call the sign shop about the aw..."
5,1,6,1,"Sure, that is mine."
6,1,7,1,"Agreed, let us keep it short today."
7,1,8,1,And the mailbox at the old address still has t...
8,1,9,1,Should we put a neon sign in the window?
9,1,10,1,"No, we already decided against it."


In [10]:
print("Transcript segments:", len(transcripts_df))
print("Total sentences:", len(sentences_df))
print("Average sentences per segment:", round(
    transcripts_df["sentence_count"].mean(), 2
))
print("Total characters:", transcripts_df["clean_text"].str.len().sum())

Transcript segments: 154
Total sentences: 157
Average sentences per segment: 1.02
Total characters: 6121


In [11]:
for _, row in sentences_df.head(15).iterrows():

    print(
        f"Meeting {row['meeting_id']} | "
        f"Reply {row['reply_id']} | "
        f"Sentence {row['sentence_id']}"
    )

    print(row["sentence"])
    print("-" * 80)

Meeting 1 | Reply 1 | Sentence 1
Right, let us get started, I have a hard stop at eleven.
--------------------------------------------------------------------------------
Meeting 1 | Reply 2 | Sentence 1
Two people are off next week, so we will be short handed.
--------------------------------------------------------------------------------
Meeting 1 | Reply 3 | Sentence 1
I will update the pricing on the kiosk by Wednesday.
--------------------------------------------------------------------------------
Meeting 1 | Reply 4 | Sentence 1
I will take down the summer banner.
--------------------------------------------------------------------------------
Meeting 1 | Reply 5 | Sentence 1
Nadia, can you call the sign shop about the awning?
--------------------------------------------------------------------------------
Meeting 1 | Reply 6 | Sentence 1
Sure, that is mine.
--------------------------------------------------------------------------------
Meeting 1 | Reply 7 | Sentence 1
Agreed,

In [12]:
OUTPUT_PATH = BASE_DIR / "data" / "processed_transcripts.csv"

sentences_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: c:\Users\Anshu\Documents\Zeravia AI projects\AI Meeting\data\processed_transcripts.csv
